# Cases

In [ ]:
import json

import pandas as pd

from recon import check, db, gap, identity, intent, jobs, observe, processing, queue, storage
from recon.execution import JobStatus

s3 = storage.get_s3_client()
BUCKET, _ = storage.parse_s3_path(storage.model_base_path(0))
DOMAIN = "N10S10E10W10"
LAKE = "testlake"
ND_FOLDER = "nd=1.0E03"


class StandInRunner:
    """Accepts submissions and reports whatever status a case needs.

    Stands in for SEPEX. It writes no artifacts — cases place those in storage
    themselves, which is the honest way to test a loop that believes storage
    over jobs. No reap(): SEPEX owns its job records, and DELETE /jobs there
    means *dismiss a running job*, not "discard a finished one".
    """

    def __init__(self):
        self.submitted, self.status = [], JobStatus.RUNNING

    def submit(self, process_id, payload, tags=None):
        reach_id = payload.get("reach_id") or int(
            payload["model_manifest_path"].split("/reach=")[1].split("/")[0])
        self.submitted.append((process_id, reach_id))
        return f"stand-in-{reach_id}-{len(self.submitted)}"

    def poll(self, ref):
        return self.status

    def logs(self, ref, tail=50):
        return "stand-in failure"


LULC = {"11": 0.04, "21": 0.04}

def reset(*reach_ids, water_body=True):
    """Terminal reaches with no dependencies, defaults seeded, storage clean.

    Terminal because this notebook is about one reach at a time; the cascade is
    `03_run_network.ipynb`'s story. `water_body=False` stages the one terminal
    that cannot run ND at all — it drains nowhere the system knows about.
    """
    with db.connect() as conn:
        conn.execute("TRUNCATE reach_network, lakes CASCADE")
        conn.execute("DELETE FROM desired_state_defaults")
        conn.execute(
            """INSERT INTO desired_state_defaults
               (sdr_commit, grid_resolution, epsg_code, dem_source, lulc_source, lulc_lookup,
                solver, q_lower_bound, q_upper_bound, initial_dq_step_for_nd)
               VALUES ('deadbeefcafe', 10, 5070, 's3://dem', 's3://lulc', %s,
                       'lisflood', 10, 100, 10)""",
            (json.dumps(LULC),))
        conn.execute(
            "INSERT INTO lakes (lake_id, geom) VALUES (%s, "
            "ST_GeomFromText('MULTIPOLYGON(((0 0,0 1,1 1,1 0,0 0)))', 5070))", (LAKE,))
        for rid in reach_ids:
            wkt = f"LINESTRING(0 0,{rid} 1)"
            conn.execute(
                "INSERT INTO reach_network (reach_id, is_terminal, terminal_reason, lake_to_id,"
                " geom) VALUES (%s, TRUE, %s, %s, ST_GeomFromText(%s, 5070))",
                (rid, "lake" if water_body else "outlet", LAKE if water_body else None, wkt))
        conn.execute("INSERT INTO desired_state (reach_id) SELECT reach_id FROM reach_network")
    s3.put_object(Bucket=BUCKET,
                  Key=storage.parse_s3_path(storage.boundary_polygon_path("lake", LAKE))[1],
                  Body=json.dumps({"type": "FeatureCollection", "features": []}).encode())
    for rid in reach_ids:
        clear_storage(rid)
    return StandInRunner()


def clear_storage(reach_id, models=True, runs=True):
    """Remove a reach's artifacts. Models and runs separately, so a case can
    delete a library without disturbing the model it was run against."""
    prefixes = []
    if models:
        prefixes.append(storage.parse_s3_path(storage.model_base_path(reach_id))[1])
    if runs:
        prefixes.append(f"version=v1/results/reach={reach_id}")
    for prefix in prefixes:
        for obj in s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix).get("Contents", []):
            s3.delete_object(Bucket=BUCKET, Key=obj["Key"])


def put_model(reach_id, manifest=True, break_identity=False):
    """Stage what a finished build leaves behind — at the PREDICTED address,
    with a manifest that passes verification (unless asked to break it)."""
    wanted = intent.effective(reach_id)
    identity_obj, ihash = identity.model_identity(wanted)
    if break_identity:
        identity_obj = {**identity_obj, "grid_resolution": 999.0}
    model_id = f"{ihash}_{DOMAIN}"
    _, base = storage.parse_s3_path(storage.model_base_path(reach_id))
    s3.put_object(Bucket=BUCKET, Key=f"{base}/{model_id}/dem.tif", Body=b"raster")
    if manifest:
        s3.put_object(
            Bucket=BUCKET, Key=f"{base}/{model_id}/{storage.MANIFEST_FILENAME}",
            Body=json.dumps({"reach_id": reach_id, "identity_hash": ihash,
                             "identity": identity_obj, "model_id": model_id,
                             "created_at": "2026-08-19T00:00:00Z"}).encode())
    return model_id


def put_nd_library(reach_id, discharges=None, manifest=True):
    """Stage a normal-depth library at the predicted address.

    `discharges` defaults to a set that spans the authored range. Pass a
    narrower one to stage a library that does not satisfy intent.
    """
    wanted = intent.effective(reach_id)
    _, mhash = identity.model_identity(wanted)
    run_obj, rhash = identity.run_identity(wanted)
    model_id = f"{mhash}_{DOMAIN}"
    library = f"{storage.nd_run_base_path(reach_id, model_id, rhash)}/{ND_FOLDER}"
    _, base = storage.parse_s3_path(library)
    if discharges is None:
        discharges = [wanted["q_lower_bound"], 55, wanted["q_upper_bound"]]
    for q in discharges:
        folder = f"{base}/{identity.q_folder(q)}"
        s3.put_object(Bucket=BUCKET, Key=f"{folder}/{storage.INUNDATED_AREA_FILENAME}",
                      Body=b'{"type":"FeatureCollection","features":[]}')
        if manifest:
            s3.put_object(
                Bucket=BUCKET, Key=f"{folder}/{storage.SCENARIO_MANIFEST_FILENAME}",
                Body=json.dumps({"reach_id": reach_id, "identity": run_obj,
                                 "identity_hash": rhash, "model_id": model_id,
                                 "inputs": {"us_discharge": float(q)},
                                 "properties": {"nominal_wse": 200.0 + q / 10}}).encode())
    return library


def put_everything(reach_id):
    """A reach whose model and library both already exist."""
    put_model(reach_id)
    put_nd_library(reach_id)


def state_of(reach_id):
    return db.one("SELECT state, model_id, nd_materialized, nd_discharges, "
                  "model_applied_revision, nd_applied_revision, desired_revision, has_gap "
                  "FROM reach_status WHERE reach_id = %s", (reach_id,))


print("ready")

## Case 1

In [ ]:
runner = reset(1)
put_everything(1)

print("check:", check.run_check(1, runner))
print("state:", state_of(1))
print("jobs submitted:", runner.submitted)

## Case 2

In [ ]:
runner = reset(2)

print("check 1:", check.run_check(2, runner))
print("check 2:", check.run_check(2, runner), "  <- already in flight")

runner.status = JobStatus.SUCCEEDED
put_model(2)
jobs.poll_in_flight(runner)
print("check 3:", check.run_check(2, runner), "  <- model adopted, nd submitted")

put_nd_library(2)
jobs.poll_in_flight(runner)
print("check 4:", check.run_check(2, runner))
print("check 5:", check.run_check(2, runner))

print(f"\nsubmissions: {runner.submitted}")
print("state:      ", state_of(2))
print("still due:  ", [r["reach_id"] for r in queue.due_reaches()])

## Case 3

In [ ]:
runner = reset(3)
put_model(3, manifest=False)

print("observe:", observe.observe_reach(3))
print("check:  ", check.run_check(3, runner))

## Case 4

In [ ]:
runner = reset(4)
put_model(4, break_identity=True)

seen = observe.observe_reach(4)
print("adopted:", seen["found"])
print("refused:", json.dumps(seen["refused"], indent=2))
print("check:  ", check.run_check(4, runner))

## Case 5

In [ ]:
runner = reset(5)
put_model(5)
put_nd_library(5, discharges=[10, 40])
observe.observe_reach(5)

seen = observe.observe_nd_runs(5)
print("adopted:", seen["found"], "|", seen["note"])
print("check:  ", check.run_check(5, runner))

put_nd_library(5)
print("\nonce it spans:", observe.observe_nd_runs(5)["q_set"])
print("check:  ", check.run_check(5, runner))
print("state:  ", state_of(5))

## Case 6

In [ ]:
runner = reset(6)
put_everything(6)
check.run_check(6, runner)
print("before:", state_of(6))

clear_storage(6, runs=False)
queue.request_check(6)

print("after: ", check.run_check(6, runner))
print("state: ", state_of(6))
print("nd row:", db.one("SELECT count(*) AS rows FROM materialized_nd_runs WHERE reach_id = 6"))

## Case 7

In [ ]:
runner = reset(7)
put_everything(7)
check.run_check(7, runner)
print("satisfied:      ", state_of(7))

with db.connect() as conn:
    conn.execute("UPDATE desired_state_defaults SET grid_resolution = 30")
print("intent changed: ", check.run_check(7, runner), "| submissions:", len(runner.submitted))

with db.connect() as conn:
    conn.execute("UPDATE desired_state_defaults SET grid_resolution = 10")
print("reverted:       ", check.run_check(7, runner), "| submissions:", len(runner.submitted))
print("state:          ", state_of(7))

## Case 8

In [ ]:
runner = reset(8)
put_everything(8)
check.run_check(8, runner)
print("before:", state_of(8))

with db.connect() as conn:
    conn.execute("UPDATE desired_state_defaults SET solver = 'sfincs'")

result = check.run_check(8, runner)
print("after: ", result)
print("state: ", state_of(8), "  <- model still proved, nd is not")
print("submitted:", runner.submitted, "  <- nothing new: sfincs has no image")

## Case 9

In [ ]:
runner = reset(9, water_body=False)
put_model(9)

print("check:", check.run_check(9, runner))
print("state:", state_of(9))
print("submissions:", runner.submitted, "| failures:",
      db.one("SELECT consecutive_failures, halted FROM reach_processing WHERE reach_id = 9"))

## Case 10

In [ ]:
class BrokenRunner(StandInRunner):
    def submit(self, job, payload):
        raise RuntimeError("docker daemon unreachable")

reset(10)
broken = BrokenRunner()
rows = []
for attempt in range(6):
    check.run_check(10, broken)
    row = db.one("SELECT consecutive_failures, halted FROM reach_processing WHERE reach_id = 10")
    rows.append({"attempt": attempt + 1, **row,
                 "due": any(r["reach_id"] == 10 for r in queue.due_reaches())})
    with db.connect() as conn:
        conn.execute("UPDATE reach_processing SET next_retry_at = NULL WHERE reach_id = 10")

display(pd.DataFrame(rows))
processing.clear_halt(10)
print("after clear_halt, due:", [r["reach_id"] for r in queue.due_reaches()])

## Case 11

In [ ]:
runner = reset(11)
check.run_check(11, runner)
runner.status = JobStatus.UNKNOWN

print("still young:", jobs.poll_in_flight(runner)[0]["action"])
with db.connect() as conn:
    conn.execute("UPDATE reach_processing SET current_step_started_at = now() - interval '30 min'"
                 " WHERE reach_id = 11")
print("past grace: ", jobs.poll_in_flight(runner)[0]["action"])
print("failures:   ", db.one("SELECT consecutive_failures FROM reach_processing WHERE reach_id = 11"))
print("next check: ", check.run_check(11, runner))

## Case 12

In [ ]:
runner = reset(21, 22, 23)
rounds = []
for n in range(1, 6):
    if n == 2:
        runner.status = JobStatus.SUCCEEDED
        for rid in (21, 22, 23):
            put_model(rid)
    if n == 3:
        for rid in (21, 22, 23):
            put_nd_library(rid)
    jobs.poll_in_flight(runner)
    results = check.sweep(runner)
    rounds.append({"sweep": n, "checked": len(results),
                   "decisions": ", ".join(sorted({r.decision for r in results})) or "-",
                   "submitted_total": len(runner.submitted)})
display(pd.DataFrame(rounds))
display(pd.DataFrame(db.query(
    "SELECT reach_id, state, model_id, nd_discharges FROM reach_status ORDER BY reach_id")))

In [ ]:
with db.connect() as conn:
    conn.execute("TRUNCATE reach_network, lakes CASCADE")
    conn.execute("DELETE FROM desired_state_defaults")
for rid in (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 21, 22, 23):
    clear_storage(rid)
print("cleaned up")

## Summary